# 상생의 손 CLIP RAG + VLM

1. 레퍼런스 이미지 CLIP 인덱스 생성  
2. 현재 드론 프레임으로 `기준 사진` 검색  
3. 현재 프레임과 기준 사진을 VLM에 입력해 드론 이동 지시 생성

In [ ]:
from pathlib import Path
import json
import sys

# 프로젝트 루트와 models 폴더 어디에서 노트북을 열어도 import 가능하게 설정
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.vlm_rag import (
    DEFAULT_MODEL,
    DEFAULT_VLM_MODEL,
    build_clip_index,
    generate_drone_direction,
    retrieve_reference_images,
)

DATASET_DIR = Path(r"C:\Users\김동준\Desktop\취업준비\포스코 빅데이터 아카데미\추론용 상생의 손")
INDEX_PATH = PROJECT_ROOT / "clip_reference_embeddings.npy"
TAGS_PATH = PROJECT_ROOT / "data" / "hand_of_harmony_tags.tsv"

# 실제 드론 프레임 경로로 교체하세요.
QUERY_IMAGE = DATASET_DIR / "구도 변경 + 의미 유사" / "618709465_18356949742161605_1238918616436252702_n.jpg"
REFERENCE_CATEGORY = "기준 사진"
TOP_K = 1

print("dataset:", DATASET_DIR)
print("query:", QUERY_IMAGE)
print("CLIP:", DEFAULT_MODEL)
print("VLM:", DEFAULT_VLM_MODEL)

## 1. CLIP 인덱스 준비

인덱스가 이미 있으면 재사용합니다. 데이터셋이나 태그가 바뀌면 `FORCE_REBUILD = True`로 실행하세요.

In [ ]:
FORCE_REBUILD = False

if FORCE_REBUILD or not INDEX_PATH.exists() or not INDEX_PATH.with_suffix(".json").exists():
    index_report = build_clip_index(
        DATASET_DIR,
        INDEX_PATH,
        model_name=DEFAULT_MODEL,
        batch_size=32,
        tags_path=TAGS_PATH,
    )
    print(json.dumps(index_report, ensure_ascii=False, indent=2))
else:
    metadata = json.loads(INDEX_PATH.with_suffix(".json").read_text(encoding="utf-8"))
    print(f"기존 인덱스 사용: {metadata['count']}장 / {metadata['dimension']}차원 / 태그 {metadata.get('tagged_count', 0)}장")

## 2. 현재 프레임으로 기준 사진 검색

In [ ]:
references = retrieve_reference_images(
    QUERY_IMAGE,
    INDEX_PATH,
    top_k=TOP_K,
    reference_category=REFERENCE_CATEGORY,
)

print(json.dumps(references, ensure_ascii=False, indent=2))

In [ ]:
from IPython.display import display
from PIL import Image

print("현재 드론 프레임")
display(Image.open(QUERY_IMAGE))

for reference in references:
    print(
        f"레퍼런스 {reference['rank']} | "
        f"유사도={reference['similarity']:.4f} | "
        f"태그={', '.join(reference['tags']) or '없음'}"
    )
    display(Image.open(reference["image"]))

## 3. VLM 이동 지시 생성

CPU에서는 수 분이 걸릴 수 있습니다. 실행하려면 `RUN_VLM = True`로 변경하세요.

In [ ]:
RUN_VLM = False

if RUN_VLM:
    drone_direction = generate_drone_direction(
        QUERY_IMAGE,
        references,
        vlm_model_name=DEFAULT_VLM_MODEL,
        max_new_tokens=128,
    )
    print(drone_direction)
else:
    print("VLM 생성을 실행하려면 RUN_VLM = True로 바꾼 뒤 이 셀을 다시 실행하세요.")

## 4. 결과 JSON 저장

In [ ]:
result = {
    "query_image": str(QUERY_IMAGE.resolve()),
    "reference_category": REFERENCE_CATEGORY,
    "references": references,
    "vlm_model": DEFAULT_VLM_MODEL if RUN_VLM else None,
    "drone_direction": drone_direction if RUN_VLM else None,
}

OUTPUT_PATH = PROJECT_ROOT / "drone_rag_result.json"
OUTPUT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("저장 완료:", OUTPUT_PATH)